In [1]:
!pip install -q datasets
!pip install -U -q dspy

In [2]:
import os

# NOTE: you'll need to have a HuggingFace Pro and OpenAI subscription tokens to run this. It will cost a few cents to run the entire notebook.
hf_token = os.getenv('HF_TOKEN')
oai_token = os.getenv('OAI_TOKEN')


In [3]:
import dspy

# Model for prompt optimization/training
lm_train = dspy.LM("openai/gpt-4o-mini", api_key=oai_token, max_tokens=3000)

# Target model that will execute optimized prompts
lm_task = dspy.LM("huggingface/meta-llama/Meta-Llama-3.1-8B-Instruct", api_key=hf_token, max_tokens=3000)
dspy.configure(lm=lm_task)

In [4]:
import random
from typing import Literal

class MMLUSignature(dspy.Signature):
    """Answer a multiple choice question."""
    subject = dspy.InputField()
    question = dspy.InputField()
    choices = dspy.InputField()
    
    answer: Literal['A', 'B', 'C', 'D'] = dspy.OutputField()


class MMLUMultipleChoiceModule(dspy.Module):
        
    temperatures = [1.0, 0.95, 0.85, 0.75]

    def __init__(self):
        self.predictor = dspy.ChainOfThought(
            MMLUSignature,
            # rationale_type=dspy.OutputField(desc="Let's think step by step to determine the right outputs.")
        )

    def forward(self, subject, question, choices, answer=None):
        kwargs = dict(subject=subject, question=question, choices=choices)
        
        for temp in self.temperatures:
            try:
                return self.predictor(**kwargs, config=dict(temperature=temp))
            except Exception as e:
                print(f" Excessive output with temperature {temp}, changing temperature...")
        
        print("All attempts failed, making random guess")
        guess_idx = random.randint(0, 3)
        return dspy.Prediction(
            reasoning="Random guess",
            answer=['A', 'B', 'C', 'D'][guess_idx]
        )

In [5]:
from dspy.datasets import DataLoader

def prepare_mmlu_dataset(split, subject):

    kwargs = dict(
        fields=('subject', 'question', 'choices', 'answer'),
        input_keys=('subject', 'question', 'choices'),
        split=split,
        trust_remote_code=True
    )
    
    data = DataLoader().from_huggingface("cais/mmlu", subject, **kwargs)

    dataset = [
        dspy.Example(
            subject=x.subject.replace("_", " ").title(),
            question=x.question,
            choices='\n'.join([f"{letter}. {c}" for letter, c in zip("ABCD", x.choices)]),
            answer=['A', 'B', 'C', 'D'][x.answer]
        ).with_inputs("subject", "question", "choices") for x in data
    ]

    print(f"{split}: prepared {len(dataset)} examples about {subject}")

    return dataset

In [6]:
def optimize(program, trainset, valset, metric, optimizer_model):
    optimizer = dspy.MIPROv2(
        metric=metric, 
        prompt_model=optimizer_model, 
        auto="medium", 
        max_bootstrapped_demos=5,
        max_labeled_demos=0, 
        teacher_settings=dict(lm=optimizer_model)
    )

    return optimizer.compile(program, trainset=trainset, valset=valset, requires_permission_to_run=False)

In [ ]:
program = MMLUMultipleChoiceModule()
metric = dspy.evaluate.answer_exact_match

subject = 'all'
trainset = prepare_mmlu_dataset('dev', subject)
valset = prepare_mmlu_dataset('validation', subject)

random.Random(0).shuffle(trainset)
random.Random(0).shuffle(valset)

print(f"Optimizing program for subject: {subject}; trainset: {len(trainset)} valset: {len(valset)}")
optimized_program = optimize(program, trainset, valset, metric, lm_train)

dev: prepared 285 examples about all


2024/12/11 22:10:20 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING MEDIUM AUTO RUN SETTINGS:
num_trials: 25
minibatch: True
num_candidates: 19
valset size: 300

2024/12/11 22:10:20 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2024/12/11 22:10:20 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2024/12/11 22:10:20 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=19 sets of demonstrations...


validation: prepared 1531 examples about all
Optimizing program for subject: all; trainset: 285 valset: 1531
Bootstrapping set 1/19
Bootstrapping set 2/19


  0%|          | 1/285 [00:00<00:06, 40.97it/s]


Bootstrapped 1 full traces after 1 examples for up to 1 rounds, amounting to 1 attempts.
Bootstrapping set 3/19


...

Bootstrapped 3 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.


2024/12/11 22:10:21 INFO dspy.teleprompt.mipro_optimizer_v2: 
Proposing instructions...

2024/12/11 22:12:46 INFO dspy.teleprompt.mipro_optimizer_v2: Proposed Instructions for Predictor 0:

2024/12/11 22:12:46 INFO dspy.teleprompt.mipro_optimizer_v2: 0: Answer a multiple choice question.

2024/12/11 22:12:46 INFO dspy.teleprompt.mipro_optimizer_v2: 1: Utilize your critical thinking skills to analyze the following multiple-choice question. Carefully consider the subject matter, evaluate each answer choice, and provide a well-reasoned justification for your selected answer.

2024/12/11 22:12:46 INFO dspy.teleprompt.mipro_optimizer_v2: 2: Given the subject, question, and choices provided, analyze the options step-by-step to determine the correct answer. Provide your reasoning clearly and select the answer from the choices A, B, C, or D.

2024/12/11 22:12:46 INFO dspy.teleprompt.mipro_optimizer_v2: 3: You are in a high-stakes academic competition where you must answer multiple-choice quest

Average Metric: 177.00 / 300 (59.0%): 100%|██████████| 300/300 [00:01<00:00, 156.76it/s]


2024/12/11 22:12:48 INFO dspy.evaluate.evaluate: Average Metric: 177 / 300 (59.0%)
2024/12/11 22:12:48 INFO dspy.teleprompt.mipro_optimizer_v2: Default program score: 59.0

2024/12/11 22:12:48 INFO dspy.teleprompt.mipro_optimizer_v2: ==> STEP 3: FINDING OPTIMAL PROMPT PARAMETERS <==
2024/12/11 22:12:48 INFO dspy.teleprompt.mipro_optimizer_v2: We will evaluate the program over a series of trials with different combinations of instructions and few-shot examples to find the optimal combination using Bayesian Optimization.

/Users/charlesfeinn/miniforge-pypy3/envs/evaluate/lib/python3.9/site-packages/optuna/_experimental.py:30: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
2024/12/11 22:12:48 INFO dspy.teleprompt.mipro_optimizer_v2: == Minibatch Trial 1 / 25 ==


  0%|          | 0/25 [00:00<?, ?it/s] Excessive output with temperature 1.0, changing temperature...
 Excessive output with temperature 1.0, changing temperature...
 Excessive output with temperature 1.0, changing temperature...
 Excessive output with temperature 1.0, changing temperature...
 Excessive output with temperature 1.0, changing temperature...
 Excessive output with temperature 0.95, changing temperature...
 Excessive output with temperature 0.95, changing temperature...
 Excessive output with temperature 0.95, changing temperature...
Average Metric: 3.00 / 6 (50.0%):  24%|██▍       | 6/25 [01:50<02:56,  9.27s/it]

In [ ]:
from dspy.evaluate import Evaluate

def evaluate(dataset, program, metric):
    evaluator = dspy.Evaluate(devset=dataset, num_threads=16, metric=metric, display_progress=True, display_table=5)
    return evaluator(program, metric=metric)

In [ ]:
import json

scores = []  # Track per-subject accuracy scores
total_right = []
total_questions = []

subjects = ['abstract_algebra', 'anatomy', 'astronomy', 'business_ethics', 'clinical_knowledge', 'college_biology', 'college_chemistry', 'college_computer_science', 'college_mathematics', 'college_medicine', 'college_physics', 'computer_security', 'conceptual_physics', 'econometrics', 'electrical_engineering', 'elementary_mathematics', 'formal_logic', 'global_facts', 'high_school_biology', 'high_school_chemistry', 'high_school_computer_science', 'high_school_european_history', 'high_school_geography', 'high_school_government_and_politics', 'high_school_macroeconomics', 'high_school_mathematics', 'high_school_microeconomics', 'high_school_physics', 'high_school_psychology', 'high_school_statistics', 'high_school_us_history', 'high_school_world_history', 'human_aging', 'human_sexuality', 'international_law', 'jurisprudence', 'logical_fallacies', 'machine_learning', 'management', 'marketing', 'medical_genetics', 'miscellaneous', 'moral_disputes', 'moral_scenarios', 'nutrition', 'philosophy', 'prehistory', 'professional_accounting', 'professional_law', 'professional_medicine', 'professional_psychology', 'public_relations', 'security_studies', 'sociology', 'us_foreign_policy', 'virology', 'world_religions']

for subject in subjects:
    testset = prepare_mmlu_dataset('test', subject)

    print(f"Evaluating subject: {subject}; testset size: {len(testset)}")

    score = evaluate(testset, optimized_program, metric)
    score_as_percent = score / 100
    scores.append(score_as_percent)  # Store subject-level score

    best_scores_file_name = "best_scores.json"
    try:
        with open(best_scores_file_name, "r") as f:
            best_scores = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        best_scores = {}
    
    if score_as_percent > best_scores.get(subject, 0):
        optimized_program.save(f"{subject}_optimized.json")
        best_scores[subject] = score_as_percent
        with open(best_scores_file_name, "w") as f:
            json.dump(best_scores, f, indent=2)

    num_questions = len(testset)
    right = int(num_questions * score_as_percent)
    total_questions.append(num_questions)
    total_right.append(right)

    print(f"Subject ({subject}) score: {score_as_percent} ({right} / {num_questions})")

print(f"Average score (macro): {sum(scores) / len(scores)}")  # Average of subject scores
print(f"Average score (micro): {sum(total_right) / sum(total_questions)}")  # Total correct / total questions

test: prepared 100 examples about abstract_algebra
Evaluating subject: abstract_algebra; testset size: 100
Average Metric: 50.00 / 98 (51.0%):  98%|█████████▊| 98/100 [04:06<00:40, 20.12s/it] Excessive output with temperature 1.0, changing temperature...
 Excessive output with temperature 1.0, changing temperature...
 Excessive output with temperature 0.95, changing temperature...
Average Metric: 50.00 / 100 (50.0%): 100%|██████████| 100/100 [10:12<00:00,  6.12s/it]

2024/12/11 14:03:49 INFO dspy.evaluate.evaluate: Average Metric: 50 / 100 (50.0%)


,subject,question,choices,example_answer,reasoning,pred_answer,answer_exact_match
0,Abstract Algebra,"Find the degree for the given field extension Q(sqrt(2), sqrt(3), ...",A. 0\nB. 4\nC. 2\nD. 6,B,"To find the degree of the field extension Q(sqrt(2), sqrt(3), sqrt...",A,
1,Abstract Algebra,"Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the index of <p> in S_5.",A. 8\nB. 2\nC. 24\nD. 120,C,"To find the index of <p> in S_5, we first need to understand what ...",C,✔️ [True]
2,Abstract Algebra,Find all zeros in the indicated finite field of the given polynomi...,"A. 0\nB. 1\nC. 0,1\nD. 0,4",D,"To find the zeros of the polynomial x^5 + 3x^3 + x^2 + 2x in Z_5, ...",D,✔️ [True]
3,Abstract Algebra,Statement 1 | A factor group of a non-Abelian group is non-Abelian...,"A. True, True\nB. False, False\nC. True, False\nD. False, True",B,"To evaluate the truth of the statements, let's analyze each one se...",C,
4,Abstract Algebra,Find the product of the given polynomials in the given polynomial ...,A. 2x^2 + 5\nB. 6x^2 + 4x + 6\nC. 0\nD. x^2 + 1,B,To find the product of the given polynomials f(x) = 4x - 5 and g(x...,B,✔️ [True]


Subject (abstract_algebra) score: 0.5 (50 / 100)
test: prepared 135 examples about anatomy
Evaluating subject: anatomy; testset size: 135
Average Metric: 94.00 / 135 (69.6%): 100%|██████████| 135/135 [01:39<00:00,  1.36it/s]

2024/12/11 14:05:32 INFO dspy.evaluate.evaluate: Average Metric: 94 / 135 (69.6%)


...

,subject,question,choices,example_answer,reasoning,pred_answer,answer_exact_match
0,Virology,How many human polyomaviruses are known at present?,A. 100\nB. 1\nC. 10\nD. unknown,A,Polyomaviruses are a family of viruses that infect humans and othe...,A,✔️ [True]
1,Virology,AIDS activism in the U.S. resulted in:,A. Food and Drug Administration reforms B. Easier access to invest...,D,AIDS activism in the U.S. played a significant role in pushing for...,D,✔️ [True]
2,Virology,"Globally, the most deaths are caused by:",A. Respiratory infections\nB. Diarrheal diseases\nC. Malaria\nD. T...,B,"According to the World Health Organization (WHO), the leading caus...",A,
3,Virology,What happens during the 'incubation period'?,A. The virus is cleared from the body by the immune system B. Viru...,B,"The incubation period is the time between exposure to a pathogen, ...",B,✔️ [True]
4,Virology,Newborn infants in countries with limited access to safe water and...,A. Exclusively formula fed for six months B. Exclusively breast fe...,B,In countries with limited access to safe water and low levels of e...,B,✔️ [True]


Subject (virology) score: 0.512 (84 / 166)
test: prepared 171 examples about world_religions
Evaluating subject: world_religions; testset size: 171
Average Metric: 142.00 / 171 (83.0%): 100%|██████████| 171/171 [05:04<00:00,  1.78s/it]

2024/12/11 18:25:44 INFO dspy.evaluate.evaluate: Average Metric: 142 / 171 (83.0%)


,subject,question,choices,example_answer,reasoning,pred_answer,answer_exact_match
0,World Religions,What can murtis be translated as?,A. Offerings\nB. Prayers\nC. Apparitions\nD. Idols,D,"In Hinduism and other Eastern traditions, a murti refers to a phys...",D,✔️ [True]
1,World Religions,Naturalists who concentrated on natural elements and processes ar...,A. Daoism\nB. Yin-Yang School\nC. Mengzi\nD. Humanistic Buddhism,B,Naturalism is a philosophical and religious perspective that empha...,A,
2,World Religions,When was the first Buddhist temple constructed in Japan?,A. 325 CE\nB. 119 CE\nC. 451 CE\nD. 596 CE,D,The introduction of Buddhism to Japan is attributed to the arrival...,D,✔️ [True]
3,World Religions,What is the name of the ten day New Year festival that celebrated...,A. Akitu\nB. Wag and Thoth\nC. Bast\nD. Nehebkau,A,The question asks about a ten-day New Year festival celebrated in ...,A,✔️ [True]
4,World Religions,Which of the following terms denote the Japanese art of flower arr...,A. Zazen\nB. Tariki\nC. Kami\nD. Ikebana,D,"To answer this question, let's break down each option in relation ...",D,✔️ [True]


Subject (world_religions) score: 0.8304 (141 / 171)
Average score (macro): 0.6999192982456138
Average score (micro): 0.6902150690784788
